## Mengambil Modul

In [ ]:
%pip install ffmpeg-python
%pip install earthengine-api
%pip install ipywidgets
%pip install geopandas

## Import Modul

In [1]:
import geemap
import ipywidgets as widgets
import re
import geopandas as gpd
from IPython.display import display
import ee
import tkinter as tk
from tkinter import filedialog

## Buat map

In [2]:
# Inisialisasi Peta
Map = geemap.Map()
Map.add_basemap("SATELLITE")
# Menampilkan peta
display(Map)

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## Buat Widget

In [ ]:
# PILIHAN SUMBER ROI
roi_source = widgets.RadioButtons(
    options=["Map", "SHP"],
    description="Sumber ROI:",
    style={'description_width': 'initial'}
)

# Widget untuk memilih tahun
tahun_mulai = widgets.IntSlider(
    description="Tahun Mulai",
    min=1984,
    max=2025,
    value=2010,
    style={'description_width': 'initial'}
)

tahun_akhir = widgets.IntSlider(
    description="Tahun Akhir",
    min=1984,
    max=2025,
    value=2025,
    style={'description_width': 'initial'}
)

# Widget untuk memasukkan judul timelapse
judul_timelapse = widgets.Text(
    description="Judul:",
    placeholder="Masukkan judul timelapse...",
    style={'description_width': 'initial'}
)

# Widget untuk memilih komposit band
band_options = {
    "Natural Color (RGB)": ["Red", "Green", "Blue"],
    "False Color (NIR, Red, Green)": ["NIR", "Red", "Green"],
    "Vegetation Index (SWIR2, NIR, Red)": ["SWIR2", "NIR", "Red"],
    "SWIR (SWIR1, NIR, Green)": ["SWIR1", "NIR", "Green"]
}

band_dropdown = widgets.Dropdown(
    options=band_options.keys(),
    description="Komposit Band:",
)

# GLOBAL VARIABLE
user_roi = None
fc = None

# Tombol untuk memilih area kajian dari peta
draw_btn = widgets.Button(description="Ambil dari Map")

# Fungsi untuk mengambil ROI dari peta
def on_draw_btn_click(b):
    global user_roi
    user_roi = Map.user_roi
    if user_roi:
        print("✅ Area kajian berhasil dipilih.")
    else:
        print("⚠ Silakan pilih area di peta terlebih dahulu.")

draw_btn.on_click(on_draw_btn_click)

# Tombol untuk memilih area kajian dari Folder SHP
draw_btn_sHP = widgets.Button(description="Ambil dari SHP")

# Fungsi tombol
def on_draw_btn_sHP_click(b):
    global fc

    root = tk.Tk()
    root.withdraw() 

    # Paksa window di depan
    root.attributes('-topmost', True)
    root.lift()
    root.focus_force()

    file_path = filedialog.askopenfilename(
        title="Pilih file SHP",
        filetypes=[("Shapefile", "*.shp")]
    )
    root.destroy()

    if file_path:
        try:
            fc = gpd.read_file(file_path)
            print("✅ SHP berhasil dibaca, jumlah fitur:", len(fc))
            # Jika pakai geemap, konversi ke ee:
            # fc_ee = geemap.geopandas_to_ee(fc)
            # simpan fc_ee global jika perlu
        except Exception as e:
            print("❌ Gagal membaca SHP:", e)
    else:
        print("Tidak ada file dipilih")

# Hubungkan tombol dengan fungsi
draw_btn_sHP.on_click(on_draw_btn_sHP_click)

# Tombol untuk membuat timelapse
timelapse_btn = widgets.Button(description="Buat Timelapse")

def on_timelapse_btn_click(b):
    global user_roi, fc

    # validasi judul
    if not judul_timelapse.value.strip():
        print("⚠ Masukkan judul timelapse terlebih dahulu.")
        return
    
     # Format judul agar aman untuk nama file (hapus karakter tidak valid)
    safe_title = re.sub(r'[^a-zA-Z0-9_-]', '_', judul_timelapse.value.strip())
    # Nama file berdasarkan judul
    output_file = f"{safe_title}.gif"      

    # Pilih ROI dari peta atau SHP
    if roi_source.value == "Map":
        if user_roi is None:
            print("⚠ ROI dari map belum dipilih")
            return
        roi = user_roi

    elif roi_source.value == "SHP":
        if fc is None:
            print("⚠ SHP belum dipilih")
            return
        
        # convert geopandas ke ee
        roi = geemap.geopandas_to_ee(fc)
        overlay_data = roi  # Overlay dengan batas SHP sendiri

    # Mengambil band yang dipilih dari dropdown
    selected_bands = band_options[band_dropdown.value]

    # Buat timelapse dengan overlay poligon
    timelapse = geemap.landsat_timelapse(
        roi,
        out_gif=output_file,
        start_year=tahun_mulai.value,
        end_year=tahun_akhir.value,
        bands=selected_bands,  # Gunakan pilihan band
        frames_per_second=2,
        title=judul_timelapse.value,  # Menggunakan input judul
        font_color="white",
        overlay_data=overlay_data,  # Tambahkan batas SHP
        overlay_color="red",  # Warna poligon
        overlay_opacity=0.5  # Transparansi poligon
    )
    
    geemap.show_image(timelapse)
    print(f"✅ Timelapse berhasil dibuat: {output_file}")

timelapse_btn.on_click(on_timelapse_btn_click)

# Menampilkan widget di layar
widgetbox = widgets.VBox([
    roi_source,
    judul_timelapse,
    tahun_mulai,
    tahun_akhir,
    band_dropdown,
    draw_btn,
    draw_btn_sHP,
    timelapse_btn
])
display(widgetbox)

Output()